In [ ]:
#linux command to download zipped data in runtime
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [ ]:
#unzip zipped data
!unzip /content/new_articles.zip -d /content/drive/MyDrive/new_articles

In [ ]:
import os

In [ ]:
#merging content of different text files present in directory, to a single text file
source_dir = '/content/drive/MyDrive/new_articles'
output_dir = '/content/drive/MyDrive/merged_articles_content.txt'
files = [f for f in os.listdir(source_dir) if f.endswith('.txt')]
for file in files:
  file_path = os.path.join(source_dir, file)
  with open(file_path, 'r') as f:
    content = f.read()
  with open(output_dir, 'a') as f:
    f.write(content)

In [ ]:
output_dir = '/content/drive/MyDrive/merged_articles_content.txt'

In [ ]:
#converting raw text to small sized chunks
with open(output_dir, 'r') as f:
  content = f.read()
def chunk_text(text,chunk_size= 200):
  words = text.split()
  chunks = []
  for i in range(len(words)-chunk_size):
    chunk = ' '.join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks
chunks = chunk_text(content)

In [ ]:
chunks[0:5]

In [ ]:
#using pre trained transformer for embedding chunks of data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embedding = model.encode(chunks)

In [ ]:
embedding[0:5]

In [ ]:
embedding.shape

In [ ]:
#mapping chunk data to index
doc_store = {
    i: chunks[i]
    for i in range(len(chunks))
}

In [ ]:
doc_store[5]

In [ ]:
!pip install faiss-cpu

In [ ]:
#implementing chunk indexing, using HNSW algorithm which helps in efficient fast search among chunks
import faiss
import numpy as np

embeddings = np.array(
    embedding
).astype('float32')

dimension = embeddings.shape[1]
index = faiss.IndexHNSWFlat(
    dimension, 32
)

index.add(embeddings)

In [ ]:
#query embedding, user query is also embedded into numerical features
query_embedding = model.encode(
    ["What is Google Pixel 7a"]
)

In [ ]:
#computing distance between chunks and user query, top 3 chunk's index and distance are shown
distances, indices = index.search(
    np.array(query_embedding).astype('float32'),
    k=3
)

In [ ]:
distances

In [ ]:
indices

In [ ]:
#retrieving top 3 similar data chunks
results = []

for idx in indices[0]:

    results.append(
        doc_store[idx]
    )

print(results)

In [ ]:
#building user defined function for whole search flow
def search(query, k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype('float32'),
        k
    )

    results = []

    for idx in indices[0]:

        results.append(doc_store[idx])

    return results

In [ ]:
search('What is Google I/O?')

In [ ]:
query = 'What is news about Databricks?'

In [ ]:
#reranking model's output with crossencoder, that compares query with each output and gives score
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

pairs = [
    [query, chunk]
    for chunk in search(query)
]

scores = reranker.predict(pairs)

In [ ]:
scores

In [ ]:
search('what is news about databricks?')

In [ ]:
#storing embedding in a numpy file
np.save(
    "/content/drive/MyDrive/myvectordb/embeddings.npy",
    embedding
)

In [ ]:
#saving chunks in a file
import json
with open("/content/drive/MyDrive/myvectordb/doc_store.json","w") as f:
  json.dump(doc_store,f)

In [ ]:
#save FAISS index
faiss.write_index(
    index,
    "/content/drive/MyDrive/myvectordb/vector.index"
)

We have built a persistent VectorDB, enabling vector search and efficient retrieval with reranking